# QIGOA Reality-Check — Kaggle runner

Notebook chạy pipeline QIGOA Reality-Check trên Kaggle. **Đọc `docs/huong-dan-setup-kaggle.md` trước.**

> 👉 **LÔ QUYẾT ĐỊNH — chạy đúng lô này, không chạy gì khác:** (a) → (c) → (d2) → (e) → (f2) → (g).
> **Một session · CPU · ~3–5 giờ · KHÔNG bật GPU.** Trả về **Bảng II** (kết quả thật đầu tiên vào bản thảo) + câu trả lời cho **3/4 câu hỏi có thể giết bài** — trước khi chi 20 giờ cho lưới chính.

**Quy tắc bất khả xâm phạm (CLAUDE.md §2):**
- Số synthetic (smoke test) **không bao giờ** là kết quả — luôn `[PLACEHOLDER]`.
- Số từ `exp_week1.yaml` là **`screening`** (n=60, 3 seed) — quyết định được, **công bố không được**.
- Mọi con số vào paper phải truy được về `results/` qua `docs/RESULTS.md`.
- **CẤM chạy E2 (lưới chính) trước** khi E1 PASS và 4 thí nghiệm rủi ro Tuần-1 còn sống (preregistration §6/A10).
- Notebook này **KHÔNG nhúng số kết quả** — nó chỉ chạy script và ghi `results/`. Output cell không phải nguồn số liệu; `results/*.csv` mới là.

**Thứ tự cell:** (a) setup + kiểm tra `/kaggle/input` → (b) cờ chọn experiment → (c) **3 cổng cứng pytest** → (d) smoke synthetic → **(d2) E0 cohort + split cấp bệnh nhân** → (e) **E1 cổng bit-exact** → **(f2) Tuần-1: 3 thí nghiệm rủi ro** → (f) E2 theo `--k-subset` *(chỉ khi lô trên pass)* → (g) zip `results/` → (h) in `run-manifest.json`.

## (a) Setup — clone code + cài package thiếu + kiểm tra dataset đã mount

Cần **Internet: ON** (Settings phải). **Accelerator: None** cho toàn bộ LÔ QUYẾT ĐỊNH — GPU chỉ dùng cho `run_unet.py` về sau (quota 30 GPU-h/tuần).

Ghi lại commit hash — nó vào `run-manifest.json` và vào dòng provenance trong `docs/RESULTS.md`.

In [ ]:
# Clone repo (Cách A). Nếu dùng utility-dataset (Cách B) thì thay bằng cp -r từ /kaggle/input/.
!git clone https://github.com/HoangLong08/Paper-DAU.git /kaggle/working/qigoa
%cd /kaggle/working/qigoa
COMMIT = !git rev-parse HEAD
print('git commit:', COMMIT[0] if COMMIT else 'UNKNOWN (not a git checkout)')

# Kaggle đã có sẵn numpy/scipy/scikit-image/scikit-learn/pandas/nibabel/PyYAML/matplotlib/torch/tqdm.
# Thường chỉ cần cài thêm medpy (HD95/NSD) + tifffile (LGG 3 kênh):
!pip install -q medpy tifffile
# Nếu medpy lỗi build: !pip install -q git+https://github.com/deepmind/surface-distance.git

In [ ]:
# Kiem tra /kaggle/input da mount + liet ke dataset (Buoc 1). /kaggle/input/ la READ-ONLY.
import os

INPUT_ROOT = '/kaggle/input'
EXPECTED = {
    'brats20-dataset-training-validation': 'CHINH (E1-E6)',
    'lgg-mri-segmentation': 'E7 - xem canh bao A8 truoc khi goi la external validation',
}

if not os.path.isdir(INPUT_ROOT):
    print('KHONG THAY /kaggle/input -> notebook nay dang chay ngoai Kaggle?')
else:
    print('Dataset dang mount tai /kaggle/input:')
    for d in sorted(os.listdir(INPUT_ROOT)):
        print('   -', d)
    print()
    for slug, role in EXPECTED.items():
        ok = os.path.isdir(os.path.join(INPUT_ROOT, slug))
        print(('OK   ' if ok else 'THIEU'), slug, '|', role)

print('\nGhi ket qua vao /kaggle/working/ (mat khi het session -> phai tai ve, cell (g)).')

## (b) Cờ chọn experiment

> 👉 **Khuyến nghị: chỉ chạy LÔ QUYẾT ĐỊNH** — cell (c) → (d2) → (e) → (f2) → (g), **một session, CPU, ~3–5h**. Nó cho **Bảng II** (số thật đầu tiên vào paper) + câu trả lời cho 3/4 câu hỏi có thể giết bài. Mọi dòng còn lại của bảng dưới **chỉ mở khi lô đó pass**. Xem `docs/huong-dan-setup-kaggle.md`.

Đặt `EXPERIMENT` rồi chạy các cell dưới. Thứ tự hợp lệ **bị ép** theo preregistration §6/A10 — cell (c), (d2), (e) là cổng cứng, không bỏ qua được.

| `EXPERIMENT` | Script | Compute | Điều kiện tiên quyết |
|---|---|---|---|
| `smoke` | `run_exact_check.py --config configs/exp_smoke.yaml` | < 60s CPU | — (số là `[PLACEHOLDER]`) |
| **`e1_exact`** | `run_exact_check.py --config configs/exp_main.yaml` | < 30ph CPU | pytest gate PASS **+ E0 (cell d2)** |
| **`week1_risky`** | `run_ceiling.py --stage p5_nested_cv` · `run_main_grid.py --config configs/exp_week1.yaml` | ~2–3h CPU | E1 PASS |
| `e2_main` | `run_main_grid.py --k-subset ...` | ~20h CPU, chia stage | E1 PASS **và** cả 4 rủi ro sống |
| `e3_ablation` | `run_ablation.py` | ~6h CPU | E2 xong |
| `e4_ceiling` | `run_ceiling.py` | ~2h CPU | E2 xong |
| `e4_unet` | `run_unet.py` | ~2h **GPU** | E4 ceiling xong |
| `e6_stats` | `run_stats.py` | phút | E2/E3/E4 xong |
| `e7_external` | `run_external.py` | ~8h CPU | ⚠️ đọc A8 trước |

In [ ]:
# ===== CO CHON EXPERIMENT =====
EXPERIMENT = 'smoke'   # smoke | e1_exact | week1_risky | e2_main | e3_ablation | e4_ceiling | e4_unet | e6_stats | e7_external
K_SUBSET   = '2,3,4'   # chi dung cho e2_main: chia stage theo k (session <= 12h)

CONFIG = {
    'smoke':        'configs/exp_smoke.yaml',
    'e1_exact':     'configs/exp_main.yaml',
    'week1_risky':  'configs/exp_week1.yaml',   # luoi SANG LOC (#3, #4). P5 (#2) dung exp_ceiling.yaml.
    'e2_main':      'configs/exp_main.yaml',
    'e3_ablation':  'configs/exp_ablation.yaml',
    'e4_ceiling':   'configs/exp_ceiling.yaml',
    'e4_unet':      'configs/exp_unet.yaml',
    'e6_stats':     'configs/exp_main.yaml',
    'e7_external':  'configs/exp_external.yaml',
}[EXPERIMENT]

print('EXPERIMENT =', EXPERIMENT, '| config =', CONFIG)
if EXPERIMENT == 'e4_unet':
    print('>> Bat Accelerator: GPU T4 cho experiment nay. Moi experiment khac de None.')
elif EXPERIMENT == 'week1_risky':
    print('>> So tu exp_week1.yaml la "screening" (n=60, 3 seed): tra loi DAU va HUONG.')
    print('   KHONG vao Bang III / Hinh 2 / Hinh 3 — bang cua paper den tu exp_main.yaml.')
elif EXPERIMENT == 'e2_main':
    print('>> CHI chay khi E1 PASS va ca 4 thi nghiem rui ro Tuan-1 con song (A10). K_SUBSET =', K_SUBSET)

## (c) 3 cổng cứng pytest — chạy TRƯỚC mọi thứ, FAIL thì DỪNG

`test_exact_dp` (DP khớp vét cạn) · `test_nfe_budget` (mọi thuật toán tiêu đúng cùng NFE, ±0) · `test_degeneracy` (unit test số học của cơ chế suy biến — **không phải** một Result).

> 🔴 Cell này **raise SystemExit** nếu bất kỳ test nào fail. Không chạy tiếp. `test_nfe_budget` fail ⇒ **lưới vô hiệu** (đó chính là lỗi đã giết lô cũ: thừa 13,4% NFE).

In [ ]:
import subprocess, sys

GATES = ['tests/test_exact_dp.py', 'tests/test_nfe_budget.py', 'tests/test_degeneracy.py']
proc = subprocess.run([sys.executable, '-m', 'pytest', *GATES, '-q'], text=True)

if proc.returncode != 0:
    raise SystemExit(
        'CONG CUNG FAIL -> DUNG TOAN BO. Khong chay bat ky experiment nao.\n'
        ' - test_exact_dp fail  : moi ket luan ve P2 dung tren DP. Debug DAU TIEN = audit quy uoc\n'
        '                         (0log0, lop rong, co tinh nen cuong-do-0 khong) — KHONG phai debug DP (A5b/A5c).\n'
        ' - test_nfe_budget fail: LUOI VO HIEU (day la loi da giet lo cu: +13,4% NFE).\n'
        ' - test_degeneracy fail: co che suy bien khong dung nhu phat bieu -> doc lai prereg §6/A1.'
    )
print('\n3 CONG CUNG PASS -> duoc phep chay experiment.')

## (d) Smoke test — synthetic, < 60s

Chạy pipeline trên dữ liệu synthetic để bắt lỗi **cấu trúc** trước khi đốt giờ Kaggle trên dữ liệu thật.

> ⚠️ **Mọi số ở cell này là `[PLACEHOLDER]`** — synthetic phantom, **KHÔNG phải kết quả**, không bao giờ vào paper, không bao giờ vào `docs/RESULTS.md` như một run thật.

In [ ]:
!python scripts/run_exact_check.py --config configs/exp_smoke.yaml
print('\n[PLACEHOLDER] So o tren la synthetic — chi xac nhan pipeline chay end-to-end.')

## (d2) E0 — cohort + split cấp BỆNH NHÂN ★ BẮT BUỘC trước P5

Sinh `data/splits/brats_cohort.csv` + `fold_{0..4}.json` — chia ở **cấp BỆNH NHÂN**, phân tầng grade × tertile thể tích WT (A3).

> 🔴 **Không bỏ qua.** `run_ceiling.py` không thấy `fold_*.json` sẽ **âm thầm rơi về chia vòng tròn** kèm một dòng `[WARN]` dễ trôi qua mắt — và P5 (cell f2) sẽ đánh giá **đóng góp dương của bài** trên split không phân tầng. Đó đúng là thứ kỷ luật mà bài này đi tố cáo người khác vi phạm.

In [ ]:
import glob, subprocess, sys

proc = subprocess.run([sys.executable, 'scripts/make_splits.py', '--config', 'configs/exp_main.yaml', '--resume'], text=True)
folds = sorted(glob.glob('data/splits/fold_*.json'))

if proc.returncode != 0 or not folds:
    raise SystemExit(
        'E0 FAIL -> DUNG. Khong co data/splits/fold_*.json thi P5 (cell f2) se chay tren\n'
        'split vong tron KHONG phan tang (A3) -> ket qua P5 khong dung duoc lam dong gop duong.'
    )
print('\nE0 OK:', len(folds), 'fold ->', ', '.join(folds))

## (f2) TUẦN 1 — thí nghiệm RỦI RO ★ chạy TRƯỚC E2 (prereg §6/A10)

Ba câu hỏi có thể **giết bài**, tốn ~2–3h. E2 (~20h) chỉ đáng chi khi cả ba còn sống.

| # | Câu hỏi | Kết quả giết bài |
|---|---|---|
| **#2** P5 nested CV | Bài có **đóng góp dương** không? | P5 thua ⇒ **không chết**: fallback đã khoá trước (prereg A10 #2) — đóng góp rơi về ceiling decomposition + công cụ chẩn đoán + checklist |
| **#3** Spearman(k, Dice) **theo từng rule** | "metric sai" hay chỉ "decoder sai"? | Tương quan âm **chỉ** dưới rule `brightest` ⇒ headline *"metrics are anti-correlated"* **SAI** |
| **#4** `morph` vs `oracle_interval` | | `morph` **vượt** oracle ⇒ **P4 tự bác bỏ** |

> **#1 (Bảng I — trắc lượng thư mục, 2 coder + Cohen's κ) KHÔNG có script** — đó chính là lý do nó dễ bị hoãn vô hạn. Khảo sát tay ~1 tuần, làm song song trong lúc Kaggle chạy. Nó quyết định P1 có mục tiêu hay đang đánh strawman.
>
> ⛔ **Số từ `exp_week1.yaml` là `screening`** (n=60, 3 seed) — trả lời **dấu và hướng**, **KHÔNG vào Bảng III/Hình 2/Hình 3**. Sát ngưỡng ⇒ không kết luận, chạy lại trên `exp_main.yaml`.

In [ ]:
# RUI RO #2 — P5 nested CV cap benh nhan (~1h). Doi hoi E0 (cell d2) da sinh fold_*.json.
!python scripts/run_ceiling.py --config configs/exp_ceiling.yaml --stage p5_nested_cv

# RUI RO #3 + #4 — luoi SANG LOC: giu tron truc k + ca 4 decoding rule, rut n va seed (~1-2h).
!python scripts/run_main_grid.py --config configs/exp_week1.yaml --resume
!python scripts/run_stats.py     --config configs/exp_week1.yaml

print('\n=> Doc results/week1/stats/ + results/ceiling/qstar.csv TRUOC khi mo E2.')
print('   Dong provenance cua lo nay PHAI ghi chu "screening" (n=60, 3 seed).')

## (e) E1 — cổng bit-exact ★ CỔNG CỨNG THỨ HAI (trên dữ liệu THẬT)

DP phải khớp vét cạn tại k=2,3 theo quy ước A5a: `|f_DP − f_brute| ≤ 1e-9` **và** mask cảm sinh giống hệt (ngưỡng canonicalise).

> 🔴 **FAIL → DỪNG TOÀN BỘ.** Cho ra **Bảng II** (`results/exact/dp_vs_bruteforce.csv`).

In [ ]:
import time
stamp = time.strftime('%Y%m%d_%H%M')
!cd /kaggle/working/qigoa && tar czf /kaggle/working/results_{stamp}.tgz results/ data/splits/
print('Da dong goi: /kaggle/working/results_' + stamp + '.tgz')
print('Tai ve tu tab Output -> giai nen vao repo -> them provenance vao docs/RESULTS.md -> commit.')

## (f) E2 — lưới chính theo `--k-subset`, resume-được

> 🔴 **CHỈ chạy khi:** E1 PASS **và** cả 4 thí nghiệm rủi ro Tuần-1 còn sống (Bảng I κ · Bậc-5 nested CV · Spearman(k,Dice) theo từng rule · morph vs oracle). Xem `docs/huong-dan-setup-kaggle.md` Bước 3.3.

Session ≤ 12h ⇒ E2 (~20h) chia theo `k`. Script checkpoint sau mỗi `(image, k, algo, seed)` vào `results/main/partial/` → session chết chỉ chạy lại phần chưa xong. GPU = None.

In [ ]:
# Chay mot stage theo K_SUBSET da dat o cell (b). Vi du: session 1 -> '2,3,4'; session 2 -> '5,6,8,10'.
assert EXPERIMENT == 'e2_main', 'Dat EXPERIMENT = "e2_main" o cell (b) truoc khi chay cell nay.'
!python scripts/run_main_grid.py --config {CONFIG} --k-subset {K_SUBSET}

# Da chay het cac k? -> merge partial + thong ke + build bang/hinh tu results/ (khong go so tay).
# !python scripts/run_stats.py --config configs/exp_main.yaml
# !python scripts/build_tables.py
# !python scripts/build_figures.py

## (g) Đóng gói `results/` để tải về

`/kaggle/working/` **mất khi session hết hạn**. Đóng gói `results/` thành `.tgz`, tải từ tab **Output**, giải nén vào repo local, **thêm dòng provenance vào `docs/RESULTS.md`**, commit. Không xoá run xấu — run âm là dữ liệu.

In [ ]:
import time
stamp = time.strftime('%Y%m%d_%H%M')
!cd /kaggle/working/qigoa && tar czf /kaggle/working/results_{stamp}.tgz results/
print('Da dong goi: /kaggle/working/results_' + stamp + '.tgz')
print('Tai ve tu tab Output -> giai nen vao repo -> them provenance vao docs/RESULTS.md -> commit.')

## (h) In `run-manifest.json`

**Không có manifest = run không tồn tại với paper** (CLAUDE.md §5.2). Manifest do `src/manifest.py` sinh cho MỖI session: `{git_commit, config_hash, seeds, dataset_version, lib_versions, timestamp, output_paths}`. Copy `git_commit` + `seeds` vào dòng provenance trong `docs/RESULTS.md`.

In [ ]:
import glob, json

manifests = sorted(glob.glob('results/**/run-manifest.json', recursive=True))
if not manifests:
    print('KHONG TIM THAY run-manifest.json trong results/.')
    print('=> Run nay KHONG TON TAI voi paper (CLAUDE.md §5.2). Kiem tra script co goi src/manifest.py khong.')
else:
    for m in manifests:
        print('=' * 70)
        print(m)
        print('=' * 70)
        print(json.dumps(json.load(open(m)), indent=2, ensure_ascii=False))
    print('\nDan dong provenance vao docs/RESULTS.md theo mau:')
    print('  Bang X <- results/<path>.csv <- scripts/<script>.py --config <config> @commit <hash>, seeds {..}')